In [8]:
import joblib

X_train = joblib.load(
    "X_train_random.pkl"
)

X_test = joblib.load(
    "X_test_random.pkl"
)

y_train = joblib.load(
    "y_train_random.pkl"
)

y_test = joblib.load(
    "y_test_random.pkl"
)

In [9]:
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    f_regression
)

In [10]:
preprocessor_compact = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "variance_filter",
        VarianceThreshold(threshold=0.05)
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "feature_selection",
        SelectKBest(
            score_func=f_regression,
            k=500
        )
    )

])

In [11]:
from lightgbm import LGBMRegressor

from xgboost import XGBRegressor

from catboost import CatBoostRegressor

from sklearn.ensemble import RandomForestRegressor

from sklearn.svm import SVR

from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.neural_network import MLPRegressor

In [12]:

# MODEL PIPELINES


lgbm_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        LGBMRegressor(
            n_estimators=300,
            random_state=42
        )
    )

])


xgb_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        XGBRegressor(
            n_estimators=300,
            random_state=42
        )
    )

])


cat_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        CatBoostRegressor(
            iterations=300,
            verbose=0,
            random_state=42,
            allow_writing_files=False
        )
    )

])


rf_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1, 
            max_depth=6
        )
    )

])

svr_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        SVR(
            kernel="rbf",
            C=10,
            epsilon=0.1
        )
    )

])

mlp_pipeline = Pipeline([
    
    ('preprocessor_compact', preprocessor_compact),
    (
        'model',MLPRegressor (
    
    
                hidden_layer_sizes = (256,128),
                activation = 'relu',
                solver = 'adam',
                alpha = 0.0001,
                batch_size = 'auto',
                learning_rate = 'adaptive',
                learning_rate_init = 0.0001,
                max_iter = 500,
                early_stopping= True,
                random_state = 42
        )
        
    )
    
])

et_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        ExtraTreesRegressor(

            n_estimators=300,

            random_state=42,

            n_jobs=-1,

            max_depth=10

        )
    )

])

hist_pipeline = Pipeline([

    ("preprocessor_compact", preprocessor_compact),

    (
        "model",
        HistGradientBoostingRegressor(

            max_iter=300,

            learning_rate=0.05,

            max_depth=6,

            random_state=42

        )
    )

])



# STORE ALL MODELS


models = {

    "LightGBM": lgbm_pipeline,

    "XGBoost": xgb_pipeline,

    "CatBoost": cat_pipeline,

    "RandomForest": rf_pipeline,
    
    "SVR": svr_pipeline,
    
    "MLPRegressor": mlp_pipeline,
    
    "ExtraTreesRegressor": et_pipeline, 
    
    "HistGradientBoostingRegressor": hist_pipeline
    

}


models = {

    "LightGBM": lgbm_pipeline,

    "XGBoost": xgb_pipeline,

    "CatBoost": cat_pipeline,

    "RandomForest": rf_pipeline,

    "ExtraTrees": et_pipeline,

    "HistGradient": hist_pipeline,

    "SVR": svr_pipeline,

    "MLPRegressor": mlp_pipeline

}

In [13]:
from sklearn.model_selection import cross_val_score

results = []

for model_name, pipeline in models.items():

    cv_scores = cross_val_score(

        pipeline,

        X_train,

        y_train,

        cv=5,

        scoring="r2",

        n_jobs=-1

    )

    results.append({

        "Model": model_name,

        "CV Mean R2": cv_scores.mean(),

        "CV Std": cv_scores.std()

    })

   
    print(model_name)
   

    print("CV Scores:")
    print(cv_scores)

    print("\nMean CV R2:")
    print(cv_scores.mean())

LightGBM
CV Scores:
[0.79082633 0.78659763 0.75288274 0.78900876 0.7602173 ]

Mean CV R2:
0.7759065531680681
XGBoost
CV Scores:
[0.76384328 0.763803   0.70727147 0.7600413  0.72958552]

Mean CV R2:
0.7449089136145085
CatBoost
CV Scores:
[0.78784977 0.78363553 0.74912234 0.78370509 0.75509967]

Mean CV R2:
0.7718824795985448
RandomForest
CV Scores:
[0.73862939 0.72065697 0.69474311 0.73271892 0.68779369]

Mean CV R2:
0.714908416974443
ExtraTrees
CV Scores:
[0.77356186 0.75465432 0.73875538 0.76244755 0.72411561]

Mean CV R2:
0.7507069447217194
HistGradient
CV Scores:
[0.79238975 0.78975818 0.75268976 0.78938563 0.75973399]

Mean CV R2:
0.7767914614361194
SVR
CV Scores:
[0.77501159 0.77737596 0.74068812 0.76213984 0.72541564]

Mean CV R2:
0.756126230882951
MLPRegressor
CV Scores:
[0.77079003 0.76527206 0.74080066 0.75938344 0.73461808]

Mean CV R2:
0.7541728506659107


In [14]:
import pandas as pd

In [15]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(

    by="CV Mean R2",

    ascending=False

)

results_df

,Model,CV Mean R2,CV Std
5,HistGradient,0.776791,0.016982
0,LightGBM,0.775907,0.016030
2,CatBoost,0.771882,0.016325
6,SVR,0.756126,0.020129
7,MLPRegressor,0.754173,0.014055
4,ExtraTrees,0.750707,0.017468
1,XGBoost,0.744909,0.022785
3,RandomForest,0.714908,0.020272


In [16]:
from sklearn.metrics import (

    r2_score,
    mean_absolute_error,
    mean_squared_error

)

In [17]:
import numpy as np

In [18]:
from sklearn.metrics import (

    r2_score,
    mean_absolute_error,
    mean_squared_error

)

test_results = []

for model_name, pipeline in models.items():

    # TRAIN MODEL
  

    pipeline.fit(
        X_train,
        y_train
    )



    
    # TRAIN PREDICTIONS
    

    y_train_pred = pipeline.predict(
        X_train
    )



    
    # TEST PREDICTIONS
    

    y_test_pred = pipeline.predict(
        X_test
    )



    
    # TRAIN METRICS
    

    train_r2 = r2_score(
        y_train,
        y_train_pred
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            y_train_pred
        )
    )



    
    # TEST METRICS
    

    test_r2 = r2_score(
        y_test,
        y_test_pred
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            y_test_pred
        )
    )

    test_mae = mean_absolute_error(
        y_test,
        y_test_pred
    )



    
    # STORE RESULTS
    

    test_results.append({

        "Model": model_name,

        "Train_R2": train_r2,

        "Test_R2": test_r2,

        "Train_RMSE": train_rmse,

        "Test_RMSE": test_rmse,

        "Test_MAE": test_mae

    })



    
    # PRINT RESULTS
    

    print("\n")
    print(model_name)
    

    print("Train R2 :", train_r2)
    print("Test R2  :", test_r2)

    print("Train RMSE :", train_rmse)
    print("Test RMSE  :", test_rmse)

    print("Test MAE :", test_mae)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.033158 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 115024
[LightGBM] [Info] Number of data points in the train set: 7984, number of used features: 500
[LightGBM] [Info] Start training from score -2.889159


c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\jeshw\anaconda3\envs\solubility_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(




LightGBM
Train R2 : 0.9451698821587413
Test R2  : 0.8082810473295221
Train RMSE : 0.5567512999743128
Test RMSE  : 1.0195196111521927
Test MAE : 0.6799239497588916


XGBoost
Train R2 : 0.9709858521665744
Test R2  : 0.7870137488174299
Train RMSE : 0.4050014330235563
Test RMSE  : 1.074580226943804
Test MAE : 0.7125710329543946


CatBoost
Train R2 : 0.8975854593587448
Test R2  : 0.7959617689080132
Train RMSE : 0.7609085442401805
Test RMSE  : 1.0517652920023541
Test MAE : 0.7069870566009765


RandomForest
Train R2 : 0.7645952521339712
Test R2  : 0.7386214190030651
Train RMSE : 1.1536106560990735
Test RMSE  : 1.1904140887031216
Test MAE : 0.8533326470343405


ExtraTrees
Train R2 : 0.870824093351658
Test R2  : 0.7780887240485395
Train RMSE : 0.854559702287436
Test RMSE  : 1.0968639057642824
Test MAE : 0.763074625833146


HistGradient
Train R2 : 0.9015578947815439
Test R2  : 0.802455470252806
Train RMSE : 0.7460056152383144
Test RMSE  : 1.0348932736594272
Test MAE : 0.695654760035307


SVR
T

In [19]:
results_df = pd.DataFrame(
    test_results
)

results_df = results_df.sort_values(

    by="Test_R2",

    ascending=False

)

results_df

,Model,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Test_MAE
0,LightGBM,0.945170,0.808281,0.556751,1.019520,0.679924
5,HistGradient,0.901558,0.802455,0.746006,1.034893,0.695655
2,CatBoost,0.897585,0.795962,0.760909,1.051765,0.706987
7,MLPRegressor,0.849183,0.789205,0.923371,1.069038,0.730382
1,XGBoost,0.970986,0.787014,0.405001,1.074580,0.712571
4,ExtraTrees,0.870824,0.778089,0.854560,1.096864,0.763075
6,SVR,0.843676,0.777621,0.940078,1.098018,0.716527
3,RandomForest,0.764595,0.738621,1.153611,1.190414,0.853333
